# Tutorial 3, Q2c: Icki-Toffee selection with maximum popularity

Release after the practical.

**The problem.** The same constraints as Q2b, but maximise the total popularity $\sum_i P_i x_i$
subject to $\sum_i W_i x_i \ge 320$ (weight, g), $\sum_i K_i x_i \le 2850$ (calories),
$\sum_i F_i x_i \le 0.45 \sum_i W_i x_i$ (fat at most 45% of the weight), $x_i \ge 0$ and whole numbers.

| Index `i` | Candy | Index `i` | Candy |
| --- | --- | --- | --- |
| 0 | Buttery butterscotch | 6 | Strawberry surprise |
| 1 | Caramel softie | 7 | Raspberry mousse |
| 2 | Orange dream | 8 | Ginger toffee |
| 3 | Coconut Ice | 9 | Rose-water posy |
| 4 | Lemon sorbet | 10 | Walnut whirl |
| 5 | Lime sherbet | 11 | Grape bunch |

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Popularity **365** with 5 Coconut Ice (`x[3]`), 50 Lime sherbet (`x[5]`) and 5 Grape
bunch (`x[11]`); the question's weight is worked out at the end.

*Original script: `5 Source Files/Python/week3_icki_toffee_q2c.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

Twelve whole-number variables: `x[i]` is the number of candies of type `i` (table above).

In [ ]:
# Integer decision variables: x[i] = number of candies of type i, >= 0 (no upper bound)
model.x = Var(range(12), domain=NonNegativeIntegers, bounds=(0, float('inf')))

Popularity of each candy type (Table 2).

In [ ]:
# Define the objective function coefficients as a list
pop = [10, 10, 8, 9, 5, 6, 7, 8, 10, 3, 8, 4]

The objective: total popularity $\sum_i P_i x_i$, maximised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(pop[i] * model.x[i] for i in range(12)), sense=maximize)

The data from Table 1: fat (g), calories and weight (g) of one candy of each type. A '-' in the table is 0.

In [ ]:
# Data for the constraints (Table 1): fat (g), calories, weight (g) per candy
fat = [20, 16, 10, 7, 6, 5, 13, 15, 18, 0, 19, 0]
cals = [100, 100, 80, 50, 60, 40, 80, 90, 80, 145, 120, 120]
weight = [24, 22, 15, 12, 10, 10, 14, 17, 20, 8, 20, 15]

The three constraints of the model. The last one is named `fat_weight_min`, but it is an **upper** limit: total fat at most 45% of total weight.

In [ ]:
# Define the constraints
model.weight_min = Constraint(expr=sum(weight[i] * model.x[i] for i in range(12)) >= 320)  # weight at least 320 g
model.cals_max = Constraint(expr=sum(cals[i] * model.x[i] for i in range(12)) <= 2850)  # calories at most 2850
model.fat_weight_min = Constraint(expr=sum(fat[i] * model.x[i] for i in range(12)) <= 0.45 * sum(weight[i] * model.x[i] for i in range(12)))  # fat at most 45% of the weight

Solve with HiGHS; `tee=True` prints the solver's log (branch and bound, because the variables are integers).

In [ ]:
# Solve the integer optimization problem
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=True)  # tee=True prints the solver's log; False hides it

If the solver reports *optimal*, print the objective and the number of each candy.

In [ ]:
# Check the solver status and print the results
if results.solver.status == SolverStatus.ok:
    if results.solver.termination_condition == TerminationCondition.optimal:
        print("Optimal solution found.")
        print("Objective value:", model.obj())
        print("Decision variables:")
        for i in range(12):
            print(f"x[{i}] =", model.x[i].value)
    else:
        print("Solver terminated with a non-optimal solution.")
else:
    print("Solver did not find a solution. Status:", results.solver.status)

## What is the weight of the selection? And what do you notice?

<details><summary>Answer</summary>

5 Coconut Ice, 50 Lime sherbet and 5 Grape bunch: weight $5 \times 12 + 50 \times 10 + 5 \times 15 = 635$ g.
Values printed as `49.99999999999984` or `5.000000000000126` are 50 and 5: tiny floating-point rounding
in the solver, not fractions of a candy.

The calories are exactly 2850, the limit, and fat is 285 g, just under $0.45 \times 635 = 285.75$ g.
The selection has 60 candies and costs 825. Total popularity grows with the *number* of candies, so the model
picks many small, low-calorie candies rather than the most popular ones: this is the flaw Q2d asks about,
and Week 04 Q1 looks at better objectives.
</details>